# OR-mask reprint segments from extractor failures

This notebook builds a **simple failure map** from the extractor:

- each frame contributes a **binary no-filament mask**;
- the masks are merged using **OR**, not vote counting;
- the expected print comes from the G-code;
- each G-code extrusion segment is scored by how much it overlaps the merged OR failure mask;
- segments whose failure severity is above a user threshold are highlighted.

This avoids overweighting regions that happened to be scanned many times.

## Step-by-step idea

### Step 1 — Extract a binary no-filament mask from each frame
For each scan image, `filament_array.extract_filament_array(...)` produces a local binary mask where the extractor believes there is **no filament**.

### Step 2 — Undo the preprocessing geometry
That local mask is put back into the original image coordinates with `undo_preprocess_mask(...)`.

### Step 3 — Merge all frame masks with OR
The global failure mask is:

```python
failure_or |= local_failure_mask
```

So a pixel is marked failed if **any frame** flagged it.

### Step 4 — Build the expected print from the G-code
The helper module rasterizes the selected G-code layer into a global expected-filament mask in the same coordinate system.

### Step 5 — Intersect expected print with failure mask
This gives a simple missed-print mask:

```python
miss_print = expected_filament & failure_or
```

### Step 6 — Score each G-code segment
For each commanded extrusion segment, sample points along the segment and look up whether those points land on the missed-print mask.

Two simple severity scores are computed:

- `severity_mean` = fraction of sampled points that are failed
- `severity_sum` = number of sampled points that are failed

`severity_mean` is usually the better threshold because it is normalized by segment length.

### Step 7 — Threshold the segment severity
Any segment with severity above your chosen threshold is highlighted as a segment that may need reprinting.

In [ ]:
import os
import math
import zipfile
import importlib.util
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

## Configuration

Keep the notebook and the three helper `.py` files in the same folder.
Only the dataset folder and G-code file usually need to be changed.

In [ ]:
# -------- Input data --------
FOLDER = r"/mnt/data/work_or/disc2accurate"   # extracted scan image folder
GCODE_FILE = r"/mnt/data/P4_one_layer_annular_disc_60OD_12ID_0p20H(3).gcode"

# Optional zip extraction if the image folder does not exist yet.
ZIP_FILE = r"/mnt/data/disc2accurate(4).zip"
EXTRACT_TO = r"/mnt/data/work_or"

# -------- Local helper modules in same folder as notebook --------
FILAMENT_ARRAY_PY = "filament_array.py"
GCODE_COORD_MASK_PY = "gcode_coordinate_mask.py"
MERGE_OR_PY = "merge_miss_print_or.py"

# -------- Extractor / merge settings --------
RADIUS = 35
THRESHOLD = 0.15
PX_PER_MM = 26.13
FLIP_X = False
FLIP_Y = True
SWAP_XY = False
LAYER_INDEX = 0
GCODE_LINE_WIDTH_MM = 0.45
MAX_FRAME_Z = 0.20
MARGIN_PX = 200

# -------- Segment severity settings --------
# Choose which severity to threshold:
#   "mean" -> fraction of sampled segment points that fall on the OR failure mask
#   "sum"  -> raw count of sampled failure points along the segment
SEVERITY_MODE = "mean"

# Main threshold to edit.
# If SEVERITY_MODE == "mean", values are usually in [0, 1].
# Example values: 0.05, 0.10, 0.15, 0.20
SEGMENT_SEVERITY_THRESHOLD = 0.10

# Line thickness used for highlighted G-code segments
SEGMENT_DRAW_THICKNESS = 2

# Where outputs will be saved
OUT_DIR = Path("./or_reprint_outputs")
OUT_DIR.mkdir(exist_ok=True)

## Optional: extract the image zip

In [ ]:
if not Path(FOLDER).exists():
    Path(EXTRACT_TO).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_FILE, 'r') as zf:
        zf.extractall(EXTRACT_TO)
    print("Extracted to:", EXTRACT_TO)
else:
    print("Folder already exists:", FOLDER)

## Load the helper modules

In [ ]:
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    import sys
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

notebook_dir = Path('.').resolve()
filament_array = load_module('filament_array', str(notebook_dir / FILAMENT_ARRAY_PY))
gcode_coordinate_mask = load_module('gcode_coordinate_mask', str(notebook_dir / GCODE_COORD_MASK_PY))
merge_miss_print_or = load_module('merge_miss_print_or', str(notebook_dir / MERGE_OR_PY))

print('Loaded:')
print(' -', notebook_dir / FILAMENT_ARRAY_PY)
print(' -', notebook_dir / GCODE_COORD_MASK_PY)
print(' -', notebook_dir / MERGE_OR_PY)

## Step 1–5: Build the OR failure mask, expected mask, and missed-print mask

This is the slow part. It runs the extractor across the folder and merges the masks with OR.

In [ ]:
out = merge_miss_print_or.merge_miss_print_or(
    folder=FOLDER,
    gcode_file=GCODE_FILE,
    radius=RADIUS,
    threshold=THRESHOLD,
    px_per_mm=PX_PER_MM,
    margin_px=MARGIN_PX,
    flip_x=FLIP_X,
    flip_y=FLIP_Y,
    swap_xy=SWAP_XY,
    layer_index=LAYER_INDEX,
    gcode_line_width_mm=GCODE_LINE_WIDTH_MM,
    max_frame_z=MAX_FRAME_Z,
    merged_out_path=str(OUT_DIR / 'merged_no_filament_or.png'),
    expected_out_path=str(OUT_DIR / 'expected_filament.png'),
    miss_out_path=str(OUT_DIR / 'miss_print_or.png'),
)

no_filament_or = out['no_filament_mask']
expected_filament = out['expected_filament_mask']
miss_print = out['miss_print_mask']
origin_x_mm, origin_y_mm = out['origin_xy_mm']

print('frames_used:', out['frames_used'])
print('origin_xy_mm:', out['origin_xy_mm'])
print('target_z:', out['target_z'])

## Visualize the three core masks

In [ ]:
def show_bool_mask(mask, title, figsize=(8, 8)):
    plt.figure(figsize=figsize)
    plt.imshow(mask, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()

show_bool_mask(no_filament_or, 'Global OR no-filament mask')
show_bool_mask(expected_filament, 'Expected filament mask from G-code')
show_bool_mask(miss_print, 'Missed-print mask = expected filament AND OR no-filament')

## Save a simple combined photo of the masks

In [ ]:
fig = plt.figure(figsize=(18, 6))
for i, (mask, title) in enumerate([
    (no_filament_or, 'OR no-filament mask'),
    (expected_filament, 'Expected filament mask'),
    (miss_print, 'Missed-print mask'),
], start=1):
    ax = fig.add_subplot(1, 3, i)
    ax.imshow(mask, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
fig.savefig(OUT_DIR / 'mask_triptych.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', OUT_DIR / 'mask_triptych.png')

## Step 6: Parse the G-code and score every extrusion segment

In [ ]:
def sample_segment_pixels(x0, y0, x1, y1, origin_x_mm, origin_y_mm, px_per_mm, shape):
    h, w = shape
    dist_mm = math.hypot(x1 - x0, y1 - y0)
    n = max(2, int(math.ceil(dist_mm * px_per_mm * 1.5)))
    t = np.linspace(0.0, 1.0, n)
    xs = x0 + (x1 - x0) * t
    ys = y0 + (y1 - y0) * t
    cols = np.round((xs - origin_x_mm) * px_per_mm).astype(int)
    rows = np.round((ys - origin_y_mm) * px_per_mm).astype(int)
    good = (rows >= 0) & (rows < h) & (cols >= 0) & (cols < w)
    return rows[good], cols[good]

segments, unsupported = gcode_coordinate_mask.parse_gcode_extrusion_segments(GCODE_FILE)
selected, target_z, layer_zs = gcode_coordinate_mask.pick_layer_segments_by_index(
    segments,
    layer_index=LAYER_INDEX,
    include_previous_layers=False,
    z_tol=0.08,
)
selected = gcode_coordinate_mask.transform_segments_xy(
    selected,
    flip_x=FLIP_X,
    flip_y=FLIP_Y,
    swap_xy=SWAP_XY,
)

segment_scores = []
for idx, seg in enumerate(selected):
    x0, y0, _, x1, y1, _, _, line_no = seg
    rr, cc = sample_segment_pixels(x0, y0, x1, y1, origin_x_mm, origin_y_mm, PX_PER_MM, miss_print.shape)
    vals = miss_print[rr, cc].astype(np.float32) if len(rr) else np.array([0.0], dtype=np.float32)
    severity_mean = float(vals.mean())
    severity_sum = float(vals.sum())
    segment_scores.append({
        'idx': idx,
        'line_no': int(line_no),
        'x0': float(x0), 'y0': float(y0),
        'x1': float(x1), 'y1': float(y1),
        'severity_mean': severity_mean,
        'severity_sum': severity_sum,
        'sample_count': int(len(vals)),
    })

print('Selected extrusion segments:', len(segment_scores))
print('Unsupported commands seen:', len(unsupported))

## Inspect severity statistics

This helps choose the threshold.

In [ ]:
severity_key = 'severity_mean' if SEVERITY_MODE == 'mean' else 'severity_sum'
vals = np.array([s[severity_key] for s in segment_scores], dtype=float)

print('Severity mode:', severity_key)
print('min  :', vals.min())
print('q25  :', np.quantile(vals, 0.25))
print('q50  :', np.quantile(vals, 0.50))
print('q75  :', np.quantile(vals, 0.75))
print('q90  :', np.quantile(vals, 0.90))
print('q95  :', np.quantile(vals, 0.95))
print('max  :', vals.max())

plt.figure(figsize=(8, 4))
plt.hist(vals, bins=50)
plt.title(f'Histogram of {severity_key}')
plt.xlabel(severity_key)
plt.ylabel('segment count')
plt.show()

## Step 7: Threshold the segment severity and render the reprint segments

Change `SEGMENT_SEVERITY_THRESHOLD` in the configuration cell, rerun that cell, then rerun this cell.

In [ ]:
def render_segment_map(expected_filament, segment_scores, severity_key, severity_threshold,
                       origin_x_mm, origin_y_mm, px_per_mm, line_thickness=2):
    img = np.zeros((*expected_filament.shape, 3), dtype=np.uint8)
    img[expected_filament] = (210, 210, 210)

    count = 0
    for s in segment_scores:
        if s[severity_key] < severity_threshold:
            continue
        c0 = int(round((s['x0'] - origin_x_mm) * px_per_mm))
        r0 = int(round((s['y0'] - origin_y_mm) * px_per_mm))
        c1 = int(round((s['x1'] - origin_x_mm) * px_per_mm))
        r1 = int(round((s['y1'] - origin_y_mm) * px_per_mm))
        cv2.line(img, (c0, r0), (c1, r1), (30, 30, 255), line_thickness, cv2.LINE_AA)
        count += 1
    return img, count

severity_key = 'severity_mean' if SEVERITY_MODE == 'mean' else 'severity_sum'
segment_img, count = render_segment_map(
    expected_filament,
    segment_scores,
    severity_key=severity_key,
    severity_threshold=SEGMENT_SEVERITY_THRESHOLD,
    origin_x_mm=origin_x_mm,
    origin_y_mm=origin_y_mm,
    px_per_mm=PX_PER_MM,
    line_thickness=SEGMENT_DRAW_THICKNESS,
)

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(segment_img, cv2.COLOR_BGR2RGB))
plt.title(f'Reprint segments from OR mask
{severity_key} >= {SEGMENT_SEVERITY_THRESHOLD} | segments={count}')
plt.axis('off')
plt.show()

cv2.imwrite(str(OUT_DIR / 'reprint_segments_from_or_mask.png'), segment_img)
print('Saved:', OUT_DIR / 'reprint_segments_from_or_mask.png')
print('Highlighted segments:', count, '/', len(segment_scores))

## Save a labeled photo with the threshold shown

In [ ]:
from PIL import Image, ImageDraw, ImageFont

base = cv2.cvtColor(segment_img, cv2.COLOR_BGR2RGB)
h, w, _ = base.shape
top = 120
canvas = np.zeros((h + top, w, 3), dtype=np.uint8)
canvas[top:] = base
pil = Image.fromarray(canvas)
draw = ImageDraw.Draw(pil)
font_big = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 42)
font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 26)

draw.text((25, 18), 'Reprint segments from OR-merged extractor failures', fill=(255, 255, 255), font=font_big)
draw.rectangle((30, 72, 80, 102), fill=(255, 40, 40))
draw.text((95, 70), f'highlighted G-code segments | {severity_key} >= {SEGMENT_SEVERITY_THRESHOLD}', fill=(235, 235, 235), font=font)
draw.rectangle((760, 72, 810, 102), fill=(210, 210, 210))
draw.text((825, 70), 'expected printed region', fill=(235, 235, 235), font=font)

labeled_path = OUT_DIR / 'reprint_segments_labeled.png'
pil.save(labeled_path)
print('Saved:', labeled_path)

## Compare several thresholds quickly

In [ ]:
thresholds = [0.05, 0.10, 0.15, 0.20] if SEVERITY_MODE == 'mean' else [5, 10, 20, 40]
fig = plt.figure(figsize=(5 * len(thresholds), 5))

for i, thr in enumerate(thresholds, start=1):
    img, count = render_segment_map(
        expected_filament,
        segment_scores,
        severity_key=severity_key,
        severity_threshold=thr,
        origin_x_mm=origin_x_mm,
        origin_y_mm=origin_y_mm,
        px_per_mm=PX_PER_MM,
        line_thickness=SEGMENT_DRAW_THICKNESS,
    )
    ax = fig.add_subplot(1, len(thresholds), i)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{severity_key} >= {thr}
segments={count}')
    ax.axis('off')

plt.tight_layout()
fig.savefig(OUT_DIR / 'reprint_segment_threshold_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', OUT_DIR / 'reprint_segment_threshold_comparison.png')

## Optional: table of the most severe segments

In [ ]:
import pandas as pd

df = pd.DataFrame(segment_scores).sort_values(severity_key, ascending=False)
df.head(25)

## Summary of what this notebook is doing

1. **Frame extractor**: build a binary no-filament mask for each frame.
2. **OR merge**: put every local mask into one global canvas and combine using OR.
3. **Expected print**: rasterize the selected G-code layer into the same global canvas.
4. **Missed print**: intersect expected print with the OR failure mask.
5. **Segment severity**: sample the missed-print mask along each commanded G-code segment.
6. **Thresholding**: highlight segments whose severity passes your chosen threshold.

This is intentionally simpler than the earlier vote-based approach, and it avoids favoring regions that were scanned more times.